# CNNFin Dataset And Image Builder

Run this notebook after `data_fetching.ipynb`. It builds the canonical `merged_df`, pickle sample splits, a balanced preview image set, and optionally the full CNN image dataset.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "cnnfin_1h.yaml").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing configs/cnnfin_1h.yaml")


ROOT = find_repo_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cnnfin.config import load_config
from cnnfin.data import align_market_data
from cnnfin.features import build_feature_table
from cnnfin.images import generate_images
from cnnfin.samples import build_samples
from image_generation.image_generator import ImageGenerator

CONFIG_PATH = ROOT / "configs" / "cnnfin_1h.yaml"
base_config = load_config(CONFIG_PATH)
artifact_dir = Path(base_config.artifact_dir)
if not artifact_dir.is_absolute():
    artifact_dir = ROOT / artifact_dir
config = load_config(CONFIG_PATH, artifact_dir=str(artifact_dir))

ARTIFACT_DIR = Path(config.artifact_dir)
RAW_DIR = ARTIFACT_DIR / "raw_candles"
PROCESSED_DIR = ARTIFACT_DIR / "processed"
REPORT_DIR = ARTIFACT_DIR / "reports"
PREVIEW_DIR = ARTIFACT_DIR / "image_preview"
FULL_IMAGE_DIR = ARTIFACT_DIR / "images"
FULL_MANIFEST_PATH = PROCESSED_DIR / "image_manifest.pkl"
FULL_MANIFEST_EXISTED_BEFORE = FULL_MANIFEST_PATH.exists()

LABEL_NAMES = {int(k): str(v) for k, v in config.class_names.items()}
IMAGE_LOOKBACK = config.image_lookback or config.lookback

print(f"Repo root: {ROOT}")
print(f"Config: {CONFIG_PATH}")
print(f"Raw candle dir: {RAW_DIR}")
print(f"Processed dir: {PROCESSED_DIR}")
print(f"Preview dir: {PREVIEW_DIR}")
print(f"Full image dir: {FULL_IMAGE_DIR}")
print(f"Model/sample lookback: {config.lookback}")
print(f"Image lookback: {IMAGE_LOOKBACK}")
print(f"Full image source size: {2 * config.image_size_each}x{2 * config.image_size_each}")

## Controls

Set `RUN_FULL_IMAGE_BUILD = True` on Jarvis when you are ready to generate all training images.

In [ ]:
RUN_FULL_IMAGE_BUILD = False
FORCE_IMAGES = False

PREVIEW_N_PER_GROUP = 3
PREVIEW_SIZE_EACH = 512
SPLITS = ["train", "val", "test"]
LABELS = [0, 1, 2]

print("RUN_FULL_IMAGE_BUILD:", RUN_FULL_IMAGE_BUILD)
print("FORCE_IMAGES:", FORCE_IMAGES)
print(f"Preview image size: {2 * PREVIEW_SIZE_EACH}x{2 * PREVIEW_SIZE_EACH}")

## Verify Raw Candles Exist

In [ ]:
raw_rows = []
for symbol in config.all_symbols:
    path = RAW_DIR / f"{symbol}_{config.interval}.pkl"
    if not path.exists():
        raise FileNotFoundError(f"Missing raw candle file for {symbol}: {path}. Run data_fetching.ipynb first.")
    df = pd.read_pickle(path)
    raw_rows.append(
        {
            "symbol": symbol,
            "path": str(path),
            "rows": int(len(df)),
            "start": str(pd.to_datetime(df["Open time"], utc=True).min()) if len(df) else None,
            "end": str(pd.to_datetime(df["Open time"], utc=True).max()) if len(df) else None,
            "duplicates": int(pd.to_datetime(df["Open time"], utc=True).duplicated().sum()),
        }
    )

raw_summary = pd.DataFrame(raw_rows)
if raw_summary["rows"].eq(0).any():
    raise ValueError("At least one raw candle file is empty.")
if raw_summary["duplicates"].gt(0).any():
    raise ValueError("At least one raw candle file has duplicate timestamps.")

display(raw_summary)

## Build Canonical Dataset

This runs the same source-of-truth modules used by the CLI pipeline.

In [ ]:
aligned = align_market_data(config)
merged_df = build_feature_table(config)
samples = build_samples(config)

print(f"aligned shape: {aligned.shape}")
print(f"merged_df shape: {merged_df.shape}")
print(f"samples shape: {samples.shape}")

## Validate Dataset Artifacts

In [ ]:
expected_paths = {
    "aligned": PROCESSED_DIR / "aligned_1h.pkl",
    "merged_pkl": PROCESSED_DIR / "merged_df.pkl",
    "samples_pkl": PROCESSED_DIR / "samples.pkl",
    "feature_columns": PROCESSED_DIR / "feature_columns.json",
    "train_samples_pkl": PROCESSED_DIR / "train_samples.pkl",
    "val_samples_pkl": PROCESSED_DIR / "val_samples.pkl",
    "test_samples_pkl": PROCESSED_DIR / "test_samples.pkl",
    "train_val_samples_pkl": PROCESSED_DIR / "train_val_samples.pkl",
    "merged_report": REPORT_DIR / "merged_df_report.json",
    "sample_report": REPORT_DIR / "sample_report.json",
}
missing_paths = [str(path) for path in expected_paths.values() if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Missing expected dataset artifacts: {missing_paths}")

missing_indicators = [col for col in config.image_indicators if col not in merged_df.columns]
if missing_indicators:
    raise ValueError(f"Missing configured image indicators: {missing_indicators}")

required_cols = [
    "BTCUSDT_Open", "BTCUSDT_High", "BTCUSDT_Low", "BTCUSDT_Close", "BTCUSDT_Volume",
    "ADAUSDT_Close", "ADAUSDT_Volume", "label", "label_status", "sample_id", "split", "valid_sample",
]
missing_required = [col for col in required_cols if col not in merged_df.columns]
if missing_required:
    raise ValueError(f"merged_df missing required columns: {missing_required}")

sample_checks = {
    "continuous_lookback": bool(samples["lookback_continuous"].all()),
    "continuous_horizon": bool(samples["horizon_continuous"].all()),
    "same_split_lookback": bool(samples["lookback_same_split"].all()),
    "same_split_horizon": bool(samples["horizon_same_split"].all()),
    "required_features_present": bool(samples["required_features_present"].all()),
    "no_ambiguous_samples": not samples["label_status"].eq("ambiguous_same_bar").any(),
}
if not all(sample_checks.values()):
    raise ValueError(f"Sample validity checks failed: {sample_checks}")

print("Label status counts:")
display(merged_df["label_status"].value_counts(dropna=False).rename("count").reset_index())
print("Sample class counts:")
display(samples.groupby(["split", "label"]).size().rename("count").reset_index())
print("Sample checks:", sample_checks)
print("Image indicators:", config.image_indicators)

## Validate Top-Left Divergence Inputs

The divergence panel must use only altcoin close columns. `BTCUSDT_Close` is excluded because it would create a fake zero row.

In [ ]:
generator = ImageGenerator(window_size=IMAGE_LOOKBACK, target_symbol=config.target_symbol)
alt_close_cols = generator._get_alt_cols(merged_df.columns, btc_close_col="Close", alt_suffix="_Close")
target_alias = f"{config.target_symbol}_Close"

if target_alias in alt_close_cols:
    raise ValueError(f"Target alias leaked into divergence panel: {target_alias}")
expected_alt_close_cols = [f"{symbol}_Close" for symbol in config.alt_symbols]
if alt_close_cols != expected_alt_close_cols:
    raise ValueError(f"Unexpected divergence columns: {alt_close_cols}")

print("Divergence panel columns:")
print(alt_close_cols)

## Build Balanced High-Resolution Preview Images

In [ ]:
def take_evenly(group: pd.DataFrame, n: int) -> pd.DataFrame:
    group = group.sort_values("Open time").reset_index(drop=True)
    if len(group) < n:
        raise ValueError(f"Need {n} rows but only found {len(group)}")
    positions = np.linspace(0, len(group) - 1, n).round().astype(int)
    return group.iloc[positions].copy()

preview_parts = []
for split in SPLITS:
    for label in LABELS:
        group = samples[(samples["split"] == split) & (samples["label"] == label)]
        preview_parts.append(take_evenly(group, PREVIEW_N_PER_GROUP))

preview = pd.concat(preview_parts, ignore_index=True)
preview["label"] = preview["label"].astype(int)
preview["label_name"] = preview["label"].map(LABEL_NAMES)
expected_preview_rows = len(SPLITS) * len(LABELS) * PREVIEW_N_PER_GROUP
assert len(preview) == expected_preview_rows, (len(preview), expected_preview_rows)

display(preview[["sample_id", "Open time", "split", "label", "label_name", "row_idx"]])

In [ ]:
PREVIEW_DIR.mkdir(parents=True, exist_ok=True)
for old_png in PREVIEW_DIR.glob("*/*/*.png"):
    old_png.unlink()

preview_manifest_rows = []
preview_generator = ImageGenerator(window_size=IMAGE_LOOKBACK, target_symbol=config.target_symbol)

for _, row in preview.iterrows():
    row_idx = int(row["row_idx"])
    start_idx = row_idx - IMAGE_LOOKBACK + 1
    split = str(row["split"])
    label = int(row["label"])
    sample_id = str(row["sample_id"])

    out_dir = PREVIEW_DIR / split / str(label)
    out_dir.mkdir(parents=True, exist_ok=True)
    image_path = out_dir / f"{sample_id}.png"

    preview_generator.save_four_panel_image(
        merged_df,
        start_idx=start_idx,
        indicators=config.image_indicators,
        filepath=str(image_path),
        size_each=PREVIEW_SIZE_EACH,
        btc_close_col="Close",
        alt_suffix="_Close",
    )

    preview_manifest_rows.append(
        {
            "sample_id": sample_id,
            "Open time": row["Open time"],
            "row_idx": row_idx,
            "start_idx": start_idx,
            "split": split,
            "label": label,
            "label_name": LABEL_NAMES[label],
            "image_lookback": IMAGE_LOOKBACK,
            "preview_size_each": PREVIEW_SIZE_EACH,
            "image_path": str(image_path),
            "image_indicators": ",".join(config.image_indicators),
        }
    )

preview_manifest = pd.DataFrame(preview_manifest_rows)
preview_manifest_path = PREVIEW_DIR / "preview_manifest.pkl"
preview_manifest.to_pickle(preview_manifest_path)
stale_preview_csv = PREVIEW_DIR / "preview_manifest.csv"
if stale_preview_csv.exists():
    stale_preview_csv.unlink()

print(f"Saved {len(preview_manifest)} preview images")
print(f"Preview manifest: {preview_manifest_path}")
display(preview_manifest.head())

## Verify Preview Images

In [ ]:
png_paths = sorted(PREVIEW_DIR.glob("*/*/*.png"))
expected_size = (2 * PREVIEW_SIZE_EACH, 2 * PREVIEW_SIZE_EACH)
if len(png_paths) != expected_preview_rows:
    raise ValueError(f"Expected {expected_preview_rows} preview PNGs, found {len(png_paths)}")

quality_rows = []
for path in png_paths:
    img = Image.open(path).convert("RGB")
    arr = np.asarray(img)
    quality_rows.append(
        {
            "image_path": str(path),
            "width": img.size[0],
            "height": img.size[1],
            "pixel_min": int(arr.min()),
            "pixel_max": int(arr.max()),
            "pixel_std": float(arr.std()),
        }
    )

quality = pd.DataFrame(quality_rows)
if not (quality[["width", "height"]] == expected_size).all().all():
    raise ValueError("At least one preview image has the wrong dimensions.")
if not (quality["pixel_std"] > 0).all():
    raise ValueError("At least one preview image appears blank.")
if int(preview_manifest["label"].eq(1).sum()) != len(SPLITS) * PREVIEW_N_PER_GROUP:
    raise ValueError("No-trade preview examples are missing.")

print(f"Verified {len(quality)} preview PNGs at {expected_size[0]}x{expected_size[1]}")
display(quality.describe(include="all"))

## Preview Grid

In [ ]:
ordered = preview_manifest.sort_values(["label", "split", "Open time"]).reset_index(drop=True)
rows = len(LABELS)
cols = len(SPLITS) * PREVIEW_N_PER_GROUP
fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.5, rows * 1.7))

for r, label in enumerate(LABELS):
    label_rows = ordered[ordered["label"] == label].reset_index(drop=True)
    for c in range(cols):
        ax = axes[r, c]
        row = label_rows.iloc[c]
        ax.imshow(Image.open(row["image_path"]))
        ax.axis("off")
        ax.set_title(str(row['split']) + "\n" + str(row['sample_id']), fontsize=7)
        if c == 0:
            ax.set_ylabel(f"{label}: {LABEL_NAMES[label]}", fontsize=10)

plt.tight_layout()
plt.show()

## Optional Full Image Build

This is the expensive Jarvis step. Leave `RUN_FULL_IMAGE_BUILD = False` for local checks.

In [ ]:
if RUN_FULL_IMAGE_BUILD:
    full_manifest = generate_images(config, force=FORCE_IMAGES)
    print(f"Full image manifest rows: {len(full_manifest):,}")
    print(f"Full image manifest: {FULL_MANIFEST_PATH}")
    if len(full_manifest) != len(samples):
        raise ValueError(f"Full manifest rows {len(full_manifest)} != samples rows {len(samples)}")
    display(full_manifest.groupby(["split", "label"]).size().rename("count").reset_index())
else:
    print("RUN_FULL_IMAGE_BUILD is False. Skipping full image generation.")
    if not FULL_MANIFEST_EXISTED_BEFORE and FULL_MANIFEST_PATH.exists():
        raise ValueError(f"Unexpected full image manifest was created: {FULL_MANIFEST_PATH}")
    if FULL_MANIFEST_PATH.exists():
        print(f"Existing full image manifest preserved: {FULL_MANIFEST_PATH}")